# Anma, Checkpoint 4: model rekomendasi dan evaluasinya
Tujuan: meranking franchise populer yang belum saya tonton, berdasarkan kemiripan cirinya dengan franchise yang sudah saya selesaikan.

**Cara menilainya (tanpa rating):** sebagian franchise yang sudah saya selesaikan disembunyikan, lalu dicek apakah model bisa menemukannya kembali di antara ratusan kandidat. Hasilnya dibandingkan dengan dua pembanding sederhana: acak dan "paling populer saja".

Metrik: **AUC** (peluang franchise tersembunyi berperingkat lebih tinggi daripada franchise acak), **P@10** (persentase tepat di 10 teratas), **R@50** (persentase franchise tersembunyi yang muncul di 50 teratas).

Catatan penting: ini masalah *positive-unlabeled*. Yang "selesai" pasti positif, tetapi yang "belum ditonton" bisa berarti tidak suka atau belum sempat.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
AKSEN, ABU = "#e0a800", "#b8b8b8"
for d in ("gambar", "hasil"):
    os.makedirs(d, exist_ok=True)

saya = pd.read_csv("data/list_saya.csv")
kat = pd.read_csv("data/katalog_populer.csv")
for df in (saya, kat):
    df["episode"] = df["episode"].replace(0, np.nan)  # MAL mengisi 0 untuk jumlah episode yang belum diketahui (mis. One Piece)
    df["panjang"] = (df["episode"] >= 100) | (df["status_tayang"] == "currently_airing")
print("List saya:", saya.shape, "| Katalog:", kat.shape)

## 1. Tabel franchise dan label

In [ ]:
wakil = kat.sort_values("peringkat_populer").drop_duplicates("franchise_id").set_index("franchise_id")
fr = wakil[["franchise_nama", "peringkat_populer", "jumlah_pengguna_list", "genre", "studio", "sumber", "tahun", "tipe"]].copy()
fr["franchise_nama"] = wakil["judul"]
fr["panjang"] = kat.groupby("franchise_id")["panjang"].any()
selesai = set(saya.loc[saya["status_list"] == "completed", "franchise_id"])
pernah = set(saya.loc[saya["status_list"] != "plan_to_watch", "franchise_id"])
rencana = set(saya.loc[saya["status_list"] == "plan_to_watch", "franchise_id"]) - pernah
fr["pernah"] = fr.index.isin(pernah)
fr = fr[~fr.index.isin(rencana)].copy()  # rencana tonton bukan sinyal yang jelas, jadi dikeluarkan
y = pd.Series(fr.index.isin(selesai).astype(int), index=fr.index, name="selesai")
print(f"Franchise: {len(fr)} | selesai (positif): {y.sum()} | lainnya (belum diketahui): {(y == 0).sum()}")

## 2. Fitur

In [ ]:
def multihot(kolom, minimal, awalan):
    daftar = fr[kolom].fillna("").str.split("|").apply(lambda l: [x for x in l if x])
    mlb = MultiLabelBinarizer()
    X = pd.DataFrame(mlb.fit_transform(daftar), columns=[f"{awalan}:{c}" for c in mlb.classes_], index=fr.index)
    return X.loc[:, X.sum() >= minimal]

def onehot(seri, minimal, awalan):
    X = pd.get_dummies(seri.fillna("tidak diketahui"), prefix=awalan, prefix_sep=":").astype(int)
    return X.loc[:, X.sum() >= minimal]

era = (fr["tahun"] // 5 * 5).astype("Int64").astype("string")
X_konten = pd.concat([
    multihot("genre", 15, "genre"), multihot("studio", 8, "studio"),
    onehot(fr["sumber"], 10, "sumber"), onehot(era, 10, "era"), onehot(fr["tipe"], 10, "tipe"),
    fr[["panjang"]].astype(int).rename(columns={"panjang": "panjang/tayang"}),
], axis=1)
log_pop = np.log10(fr["jumlah_pengguna_list"].fillna(fr["jumlah_pengguna_list"].median()))
X_populer = X_konten.assign(log_populer=(log_pop - log_pop.mean()) / log_pop.std())
print("Fitur konten:", X_konten.shape[1], "| konten + popularitas:", X_populer.shape[1])

## 3. Evaluasi
Setiap putaran: 20% franchise yang sudah saya selesaikan disembunyikan (dianggap belum diketahui), model dilatih dari sisanya, lalu diuji
apakah franchise tersembunyi itu naik ke peringkat atas di antara semua kandidat. Diulang 30 kali dengan pilihan acak yang berbeda.
Yang disembunyikan adalah satu franchise utuh, sehingga season 2 tidak "mengintip" season 1.

In [ ]:
def logreg():
    return LogisticRegression(C=0.3, class_weight="balanced", max_iter=3000)

def metrik(skor, kandidat, target):
    urut = np.argsort(-skor[kandidat])
    t = target[urut]
    return {"AUC": roc_auc_score(target, skor[kandidat]), "P@10": t[:10].mean(), "R@50": t[:50].sum() / target.sum()}

def uji(ulang=30, frac=0.2):
    pos = np.flatnonzero(y.values == 1)
    populer = fr["jumlah_pengguna_list"].fillna(0).values.astype(float)
    hasil = []
    for s in range(ulang):
        rng = np.random.default_rng(s)
        sembunyi = rng.choice(pos, size=int(len(pos) * frac), replace=False)
        y_latih = y.values.copy()
        y_latih[sembunyi] = 0
        kandidat = np.flatnonzero(y_latih == 0)
        target = np.isin(kandidat, sembunyi)
        skor = {
            "Acak": rng.random(len(fr)),
            "Paling populer": populer,
            "Konten": logreg().fit(X_konten.values, y_latih).decision_function(X_konten.values),
            "Konten + popularitas": logreg().fit(X_populer.values, y_latih).decision_function(X_populer.values),
        }
        for nama, sk in skor.items():
            hasil.append({"metode": nama, **metrik(sk, kandidat, target)})
    return pd.DataFrame(hasil)

hasil = uji()
urutan = ["Acak", "Paling populer", "Konten", "Konten + popularitas"]
rata = hasil.groupby("metode")[["AUC", "P@10", "R@50"]].mean().loc[urutan]
gal = hasil.groupby("metode")[["AUC", "P@10", "R@50"]].std().loc[urutan]
rata.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.8))
for a, k in zip(ax, rata.columns):
    a.bar(rata.index, rata[k], yerr=gal[k], color=[ABU, ABU, AKSEN, AKSEN], capsize=3)
    a.set_title(k)
    plt.setp(a.get_xticklabels(), rotation=25, ha="right")
fig.suptitle("Menemukan kembali franchise yang disembunyikan (rata-rata 30 putaran, batang galat = simpangan baku)", y=1.02)
plt.savefig("gambar/7_evaluasi_model.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Ciri apa yang dipelajari model?

In [ ]:
model = logreg().fit(X_konten.values, y.values)
koef = pd.Series(model.coef_[0], index=X_konten.columns).sort_values()
tampil = pd.concat([koef.head(10), koef.tail(10)])
ax = tampil.plot.barh(color=[ABU if v < 0 else AKSEN for v in tampil], figsize=(7, 6))
ax.set_title("Ciri yang menurunkan (abu) dan menaikkan (kuning) peluang saya menyelesaikan")
plt.savefig("gambar/8_ciri_selera.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Rekomendasi

In [ ]:
terbaik = rata.loc[["Konten", "Konten + popularitas"], "P@10"].idxmax()
X_final = X_populer if terbaik == "Konten + popularitas" else X_konten
print("Model yang dipakai untuk rekomendasi:", terbaik)

m = logreg().fit(X_final.values, y.values)
fr["skor"] = m.decision_function(X_final.values).round(2)
kontribusi = X_final.values * m.coef_[0]

def alasan(i):
    k = pd.Series(kontribusi[i], index=X_final.columns).drop(["log_populer"], errors="ignore")
    k = k[~k.index.str.startswith("tipe:")]
    return ", ".join(k[k > 0].nlargest(3).index)

fr["alasan"] = [alasan(i) for i in range(len(fr))]
belum = fr[~fr["pernah"]]
kolom = ["franchise_nama", "peringkat_populer", "skor", "alasan"]
rek_biasa = belum[~belum["panjang"]].nlargest(20, "skor")[kolom]
rek_panjang = belum[belum["panjang"]].nlargest(10, "skor")[kolom]
rek_biasa.to_csv("hasil/rekomendasi.csv", index=False)
print("20 rekomendasi (tamat atau tidak terlalu panjang):")
display(rek_biasa)
print("10 rekomendasi yang panjang atau masih tayang:")
display(rek_panjang)

## 6. Ringkasan angka

In [ ]:
print(f"Franchise di analisis: {len(fr)} | positif: {int(y.sum())}")
for nama in urutan:
    r = rata.loc[nama]
    print(f"{nama:22} AUC {r['AUC']:.2f} | P@10 {r['P@10']:.2f} | R@50 {r['R@50']:.2f}")
print(f"Model rekomendasi: {terbaik}")
print("Lima rekomendasi teratas:", ", ".join(rek_biasa["franchise_nama"].head(5)))

## 7. Temuan saya (isi sendiri, 3 sampai 5 kalimat dengan angka dari bagian 6)
- Bagian 3 adalah mencari model berdasarkan jenis pemilihan franchise, ada franchise yang saya tonton disembunyikan dan model diminta mencari franchise yang dimaksud. Hasilnya Konten + popularitas (maksud dari konten adalah ciri konten (genre, studio, sumber cerita, era)) memiliki nilai 0.81 yang artinya performanya baik. Sebenarnya konten saja sudah memiliki nilai yang cukup baik (0.74). Hal ini menandakan konten anime lebih informatif daripada popularitas. Artinya pilihan tontonan saya dipengaruhi konten daripada hanya populer. Jadi popularitas adalah penguat saya untuk menonton, bukan tujuan utama. Apabila diambil 10 anime teratas, Model K+P berpeluang menemukan 2-3 anime yang disembunyikan, dan sekitar seperempat muncul di 50 teratas (R@50 0,24). Selisih antar metode di dua angka ini kecil dibanding variasinya, jadi saya tidak menyimpulkan model gabungan pasti lebih baik di sini.
- Rekomendasi dari model memberikan 5 anime berikut : Violet Evergarden, Dr. Stone, Jujutsu Kaisen, Mob Psycho 100, dan Nichijou. Meski memiliki genre dan isi yang mirip dengan beberapa anime yang telah saya tonton, namun saya tidak minat menonton anime yang direkomendasikan murni karena mood (Belum atau tidak berkeinginan menonton). Model hanya menangkap kemiripan ciri, bukan keinginan saat itu.


**Keterbatasan:** hanya sekitar satu setengah ratus franchise positif; tanpa rating pribadi; "belum ditonton" tidak sama dengan "tidak suka"; rekomendasi hanya dari 2.000 judul populer.